# Lab 5 - BoW and TF-IDF

In [1]:
import pandas as pd
import ast
from collections import Counter
import math

df = pd.read_csv('Processed_Reviews.csv')
print(df.columns)

tokenized_reviews = df['tokenized'].dropna().apply(ast.literal_eval)

all_words = [word for review in tokenized_reviews for word in review]
word_freq = Counter(all_words)

print("Top words:")
print(pd.DataFrame(word_freq.most_common(10), columns=['Word','Freq']))

Index(['Review', 'lowercased', 'urls_removed', 'html_removed',
       'emojis_removed', 'slangs_replaced', 'contractions_replaced',
       'punctuations_removed', 'numbers_removed', 'spelling_corrected',
       'stopwords_removed', 'stemmed_words', 'lemmatized', 'tokenized'],
      dtype='object')
Top words:
        Word  Freq
0    product     7
1    quality     3
2      great     2
3    amazing     2
4       love     2
5    awesome     2
6       work     2
7  perfectly     2
8       life     2
9     expect     2


In [2]:
# Bag of Words
all_words = [word for review in tokenized_reviews for word in review]
word_freq = Counter(all_words)
sorted_word_freq = dict(sorted(word_freq.items(), key=lambda item: item[1], reverse=True))

word_freq_df = pd.DataFrame(list(sorted_word_freq.items()), columns=['Word', 'Frequency'])
print(word_freq_df.head())


      Word  Frequency
0  product          7
1  quality          3
2    great          2
3  amazing          2
4     love          2


In [3]:
# Document vectors
document_vectors = []
for review in tokenized_reviews:
    document_vector = [1 if word in review else 0 for word in sorted_word_freq.keys()]
    document_vectors.append(document_vector)

doc_vectors_df = pd.DataFrame(document_vectors, columns=sorted_word_freq.keys())
doc_vectors_df.to_csv('document_vectors.csv', index=False)
print('Document vectors saved!')


Document vectors saved!


In [4]:
# TF function
def compute_tf(document):
    word_count = Counter(document)
    return {word: count / len(document) for word, count in word_count.items()}

# IDF function
def compute_idf(documents):
    N = len(documents)
    idf = {}
    all_words = set(word for doc in documents for word in doc)
    for word in all_words:
        count = sum(1 for doc in documents if word in doc)
        idf[word] = math.log(N / count)
    return idf

# TF-IDF function
def compute_tfidf(document, idf):
    tf = compute_tf(document)
    return {word: tf[word] * idf[word] for word in tf}


In [5]:
# Compute TF, IDF, TF-IDF
documents = tokenized_reviews.tolist()

tf_data = [compute_tf(doc) for doc in documents]
tf_df = pd.DataFrame(tf_data).fillna(0)
tf_df.to_csv('tf_scores.csv', index=False)

idf = compute_idf(documents)
idf_df = pd.DataFrame([idf]).fillna(0)
idf_df.to_csv('idf_scores.csv', index=False)

tfidf_data = [compute_tfidf(doc, idf) for doc in documents]
tfidf_df = pd.DataFrame(tfidf_data).fillna(0)
tfidf_df.to_csv('tfidf_scores.csv', index=False)

print('TF, IDF, TF-IDF files saved!')


TF, IDF, TF-IDF files saved!
